<a href="https://colab.research.google.com/github/KinzaAsif2456/discoverey/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task Type: Ranking & Scoring:**

I am framing this as a Search Opportunity Scoring and Ranking problem.
Rather than a simple binary classification where every declining page gets treated equally, our content marketing team has a fixed weekly bandwidth (they can only rewrite ~20 to 50 articles per week).
Therefore, we need the model to assign a continuous refresh score (0–100) to every published page and sort the output as a ranked queue. The goal is to surface the highest-leverage pages—those with significant search volume that are losing traffic—at the top of the queue so editors spend time where revenue recovery is highest

In [25]:
import os
import sys
import subprocess
import pandas as pd
import numpy as np

In [26]:

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")


Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


In [27]:
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.head)


<bound method NDFrame.head of                  content_id          client_id  search_volume  competition  \
0      content_304f48230142  client_f369cb89fc           10.0         0.67   
1      content_a1fb4e703a9e  client_4e07408562           90.0         0.01   
2      content_9aa793d4d895  client_7f2253d7e2            0.0         0.00   
3      content_331d6c4de07b  client_19581e27de           10.0         0.00   
4      content_d99b7a2d90ca  client_3fdba35f04            0.0         0.00   
...                     ...                ...            ...          ...   
29995  content_c322796023c8  client_e29c9c180c           10.0         0.05   
29996  content_526572edb3fa  client_7f2253d7e2            0.0         0.00   
29997  content_38112bdd0c6e  client_349c41201b           10.0         1.00   
29998  content_ab26273a7e7a  client_19581e27de           10.0         0.00   
29999  content_887020f20b5e  client_6208ef0f77            0.0         0.00   

      competition_level   cpc    

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

target / Proxy Definition:
There is no explicit ground-truth column in Google Search Console that says "this page needs a refresh". Ground truth would require running an A/B test (refreshing a page and observing if traffic recovers).
Instead, we use is_declining_label as a proxy target: a page where 30-day organic click trend is negative (trend_direction == 'down'). A page experiencing sustained organic drop is our best proxy for an article needing editorial intervention.

In [28]:
# Create binary target proxy label
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# Check label distribution
decline_counts = df["is_declining_label"].value_counts()
decline_rate = df["is_declining_label"].mean()

print(f"Declining pages (Label 1): {decline_counts[1]:,} rows")
print(f"Stable/Growing pages (Label 0): {decline_counts[0]:,} rows")
print(f"Baseline decline rate in dataset: {decline_rate * 100:.1f}%")

Declining pages (Label 1): 16,262 rows
Stable/Growing pages (Label 0): 13,738 rows
Baseline decline rate in dataset: 54.2%


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary Success Metric: Precision@50
(Top-50 Precision)**



*Why Precision@50?*

I'm using Precision@50 because the two error types aren't symmetric for this decision. If a flag a page that's not actually declining, that's basically wasted editor time this week — but if I miss one, it's not really gone, it just gets re-scored next week anyway, so it's not as costly

*False Positive Cost (High):*  
Recommending a healthy page wastes 3 hours of an editor's time auditing something that didn't need fixing.

*False Negative Cost (Low):*

Missing a declining page simply means it sits in the catalog for another week until next run.
Therefore, we measure what percentage of our top-50 ranked recommendations are genuinely declining pages (is_declining_label == 1).

In [29]:
# Naive Baseline 1: Rank purely by raw Search Volume
top_50_volume = df.sort_values("search_volume", ascending=False).head(50)
p50_volume = top_50_volume["is_declining_label"].mean()

# Naive Baseline 2: Rank purely by lowest Click-Through Rate (CTR)
top_50_ctr = df.sort_values("ctr", ascending=True).head(50)
p50_ctr = top_50_ctr["is_declining_label"].mean()

print(f"Naive Rule 1 (Search Volume Alone) Precision@50: {p50_volume * 100:.1f}%")
print(f"Naive Rule 2 (Lowest CTR Alone) Precision@50:      {p50_ctr * 100:.1f}%")
print(f"Random Baseline (Dataset Average):                {decline_rate * 100:.1f}%")

Naive Rule 1 (Search Volume Alone) Precision@50: 42.0%
Naive Rule 2 (Lowest CTR Alone) Precision@50:      50.0%
Random Baseline (Dataset Average):                54.2%


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of Analysis:
1 Row = 1 Unique Published Content Item (content_id) for 1 Client (client_id) over a 90-day search window.
Each row aggregates 90 days of Google Search Console visibility metrics (impressions, clicks, average position) and GA4 engagement signals (sessions, scroll rate).

In [30]:
# Inspect the Unit of Analysis Dataframe
print(f"Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

# Display key unit identifier columns & metrics
unit_cols = [
    "content_id", "client_id", "content_type", "main_intent",
    "impressions_90d", "clicks_90d", "avg_position", "ctr",
    "content_age_days", "days_since_last_update", "is_declining_label"
]

print("\nSample Rows (Unit of Analysis):")
df[unit_cols].head(3)

Dataset Shape: 30,000 rows × 45 columns

Sample Rows (Unit of Analysis):


,content_id,client_id,content_type,main_intent,impressions_90d,clicks_90d,avg_position,ctr,content_age_days,days_since_last_update,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,3803,29,10.6,0.76,187,20,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,15320,7,20.3,0.05,445,25,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,12581,11,36.5,0.09,141,20,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A traditional heuristic rule like "flag any page over 180 days old" fails in search marketing:

**High False Positives:** Many evergreen articles stay #1 on Google for years without needing updates.

**High False Negatives:** Newly published pages (e.g. 60 days old) can decline rapidly if search intent changes or competitors publish better guides.

**Non-linear Feature Interaction:** A low CTR on Position #2 is a major opportunity, while a low CTR on Position #45 is completely normal. A static threshold cannot capture these non-linear interactions across position, impressions, and word count.

In [31]:
# Rule evaluation: Flag pages where content_age_days > 180
stale_rule_mask = df["content_age_days"] > 180
stale_pages = df[stale_rule_mask]

# Calculate Precision & Recall of the simple rule
stale_precision = stale_pages["is_declining_label"].mean() if len(stale_pages) > 0 else 0
stale_count = len(stale_pages)

print(f"Pages flagged by 'Age > 180 days' rule: {stale_count:,}")
print(f"Precision of simple rule: {stale_precision * 100:.1f}%")
print(f"Overall dataset decline rate: {decline_rate * 100:.1f}%")

Pages flagged by 'Age > 180 days' rule: 17,728
Precision of simple rule: 48.3%
Overall dataset decline rate: 54.2%


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.